# CharacterBench Evaluation on Kaggle

This notebook runs a local CharacterBench English pilot for:
- base `microsoft/Phi-3-mini-4k-instruct`
- your LoRA adapter
- CharacterJudge as the evaluator

Recommended Kaggle setup:
- GPU notebook
- internet enabled if you want to download models from Hugging Face
- otherwise upload CharacterBench, CharacterJudge, and the LoRA adapter as Kaggle datasets

The notebook loads models sequentially in 4-bit to fit Kaggle GPUs more comfortably.


In [ ]:
%pip -q install -U transformers peft accelerate bitsandbytes sentencepiece

In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Dict, List, Optional

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed
from peft import PeftModel


@dataclass
class Config:
    characterbench_dir: str = "/kaggle/input/characterbench/CharacterBench"
    base_model_id: str = "microsoft/Phi-3-mini-4k-instruct"
    adapter_path: str = "/kaggle/input/phi3-rolebench-phase1-full/phi3_rolebench_phase1_full"
    judge_model_path: str = "thu-coai/CharacterJudge"
    output_root: str = "/kaggle/working/characterbench_eval"
    only_files: List[str] = field(default_factory=lambda: ["engagement_test.json"])
    max_sessions_per_file: Optional[int] = 20
    max_new_tokens_model: int = 128
    max_new_tokens_judge: int = 256
    use_4bit: bool = True
    seed: int = 42


cfg = Config()
Path(cfg.output_root).mkdir(parents=True, exist_ok=True)
set_seed(cfg.seed)

print(json.dumps(asdict(cfg), indent=2))
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())


## Helpers


In [ ]:
def pick_dtype() -> torch.dtype:
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        return torch.bfloat16
    if torch.cuda.is_available():
        return torch.float16
    return torch.float32


def make_quant_config(use_4bit: bool) -> Optional[BitsAndBytesConfig]:
    if not use_4bit or not torch.cuda.is_available():
        return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=pick_dtype(),
    )


def unload_model(model=None):
    if model is not None:
        del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def load_causal_lm(model_id: str, adapter_path: Optional[str] = None, use_4bit: bool = True):
    tokenizer_source = adapter_path if adapter_path and Path(adapter_path).exists() else model_id
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_source, trust_remote_code=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    model_kwargs = {
        "device_map": "auto",
        "dtype": pick_dtype(),
        "trust_remote_code": False,
        "attn_implementation": "eager",
    }
    quant_config = make_quant_config(use_4bit)
    if quant_config is not None:
        model_kwargs["quantization_config"] = quant_config

    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return tokenizer, model


def generate_text(model, tokenizer, prompt: str, max_new_tokens: int) -> str:
    messages = [{"role": "user", "content": prompt}]
    if hasattr(tokenizer, "apply_chat_template"):
        rendered = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        rendered = prompt

    inputs = tokenizer(rendered, return_tensors="pt", truncation=True, max_length=3072)
    device = getattr(model, "device", None)
    if device is None:
        device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


def load_roleplay_prompt_en(characterbench_dir: Path) -> str:
    text = (characterbench_dir / "roleplay_prompt.py").read_text(encoding="utf-8")
    start = text.index("Role_Play_PROMPT_EN =")
    prompt_block = text[start:]
    first_triple = prompt_block.index('"""')
    remainder = prompt_block[first_triple + 3:]
    second_triple = remainder.index('"""')
    return remainder[:second_triple]


def norm_session_en(session: dict) -> dict:
    session_en = session["translation_en"]
    messages = []
    for msg in session_en["dialogue"]:
        speaker = msg["speaker"].lstrip()
        if speaker in {"user", "User"}:
            role = "user"
        elif speaker in {"character", "Character"}:
            role = "assistant"
        else:
            if speaker == session_en["user_name"].lstrip():
                role = "user"
            elif speaker == session_en["character_name"].lstrip() or speaker in session_en["character_name"].lstrip():
                role = "assistant"
            elif messages and messages[-1]["role"] == "user":
                role = "assistant"
            else:
                raise ValueError(f"invalid role = {speaker}")
        messages.append({"role": role, "content": msg["utterance"]})

    assert messages and messages[-1]["role"] == "user"
    greeting = session_en.get("greeting", "")
    if greeting == "N/A":
        greeting = ""
    if greeting:
        messages.insert(0, {"role": "assistant", "content": greeting})
    if messages[0]["role"] == "assistant":
        messages.insert(0, {"role": "user", "content": ""})

    return {
        "dialogue_setting": {
            "user_name": session_en["user_name"],
            "bot_name": session_en["character_name"],
            "user_profile": session_en.get("user_profile", ""),
            "bot_profile": session_en["character_profile"],
        },
        "messages": messages,
    }


def build_roleplay_prompt(roleplay_prompt_en: str, session: dict) -> str:
    dialogue = "\n".join(f"{m['role']}: {m['content']}" for m in session["messages"])
    return roleplay_prompt_en.format(
        character_profile=session["dialogue_setting"]["bot_profile"],
        user_profile=session["dialogue_setting"]["user_profile"],
        dialogue=dialogue,
    )


def extract_json_response(text: str) -> str:
    cleaned = text.replace("```json", "").replace("```", "").strip()
    try:
        payload = json.loads(cleaned)
        if isinstance(payload, dict) and "response" in payload:
            return str(payload["response"]).strip()
    except Exception:
        pass
    return cleaned.strip()


def read_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")


## Generate CharacterBench responses

This runs the benchmark model first, then unloads it, then runs the LoRA model.


In [ ]:
characterbench_dir = Path(cfg.characterbench_dir)
raw_data_dir = characterbench_dir / "eval_data" / "raw_data"
roleplay_prompt_en = load_roleplay_prompt_en(characterbench_dir)
selected_files = cfg.only_files or sorted(p.name for p in raw_data_dir.glob("*.json"))

model_specs = [
    {"label": "base", "adapter_path": None},
    {"label": "lora", "adapter_path": cfg.adapter_path},
]

response_dirs = {}
for spec in model_specs:
    label = spec["label"]
    adapter_path = spec["adapter_path"]
    response_dir = Path(cfg.output_root) / f"response_data_{label}"
    response_dir.mkdir(parents=True, exist_ok=True)
    response_dirs[label] = response_dir

    tokenizer, model = load_causal_lm(cfg.base_model_id, adapter_path=adapter_path, use_4bit=cfg.use_4bit)
    for filename in selected_files:
        raw_path = raw_data_dir / filename
        data = read_json(raw_path)
        output = []
        for idx, session in enumerate(tqdm(data, desc=f"{label}: {filename}")):
            if cfg.max_sessions_per_file is not None and idx >= cfg.max_sessions_per_file:
                break
            try:
                session_en = norm_session_en(session)
                prompt_en = build_roleplay_prompt(roleplay_prompt_en, session_en)
                response_en = extract_json_response(generate_text(model, tokenizer, prompt_en, cfg.max_new_tokens_model))
            except Exception as exc:
                print(f"Skipping {filename} idx={idx}: {exc}")
                continue

            session_copy = json.loads(json.dumps(session, ensure_ascii=False))
            session_copy.setdefault("translation_en", {})
            session_copy["translation_en"]["response_en"] = response_en
            output.append(session_copy)

        write_json(response_dir / filename, output)
        print(f"Saved {len(output)} sessions to {response_dir / filename}")

    unload_model(model)
    del tokenizer


## Convert into CharacterJudge prompts


In [ ]:
import subprocess

construct_dir = characterbench_dir / "construct_prompts"
evaluation_dirs = {}
for label, response_dir in response_dirs.items():
    evaluation_dir = Path(cfg.output_root) / f"evaluation_data_en_{label}"
    evaluation_dirs[label] = evaluation_dir
    cmd = [
        "python",
        str(construct_dir / "process_wo_context_en_all.py"),
        "--data_path", str(response_dir),
        "--output_path", str(evaluation_dir),
        "--model_name", label,
    ]
    subprocess.run(cmd, check=True, cwd=str(construct_dir))
    print(f"Built evaluation prompts in {evaluation_dir}")


## Run CharacterJudge

CharacterJudge is an 8B model. In 4-bit mode, it is usually feasible on Kaggle T4/P100 class GPUs if run sequentially; without quantization it may be tight. This is an inference from the model size and common 4-bit memory footprints, not a guarantee. The official model card lists CharacterJudge as an 8B BF16 model. Source: [CharacterJudge model card](https://huggingface.co/thu-coai/CharacterJudge)


In [ ]:
judge_tokenizer, judge_model = load_causal_lm(cfg.judge_model_path, adapter_path=None, use_4bit=cfg.use_4bit)

def read_eval_data(path: Path):
    data = read_json(path)
    rows = []
    for idx, d in enumerate(data):
        rows.append({
            "id": idx,
            "input": d["instruction"],
            "output": d["output"],
        })
    return sorted(rows, key=lambda x: len(x["input"]))


def judge_one_prompt(payload: dict) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": payload["input"]},
    ]
    if hasattr(judge_tokenizer, "apply_chat_template"):
        rendered = judge_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    else:
        rendered = payload["input"]
    return generate_text(judge_model, judge_tokenizer, rendered, cfg.max_new_tokens_judge)


def normalize_generated(generated: str):
    first_line = generated.split("\n")[0].strip()
    m = re.search(r"(\d+(?:\.\d{1,3})?)", first_line)
    if m:
        return float(m.group(1))
    return 3.0


def aggregate_scores(result_dir: Path):
    result = {}
    avg = 0.0
    files = sorted(result_dir.glob("*.jsonl"))
    for file in files:
        rows = [json.loads(line) for line in file.read_text(encoding="utf-8").splitlines() if line.strip()]
        preds = [normalize_generated(r["generated"]) for r in rows]
        golds = [float(r["output"]) for r in rows]
        mx = max(golds)
        metric = (float(np.mean(preds)) / mx) * 5
        key = file.stem.replace(file.stem.split("_")[0] + "_", "")
        result[key] = metric
        avg += metric
    result["average"] = avg / len(files) if files else 0.0
    return result


judge_result_paths = {}
summary_paths = {}
for label, evaluation_dir in evaluation_dirs.items():
    result_dir = Path(cfg.output_root) / f"judge_results_{label}"
    result_dir.mkdir(parents=True, exist_ok=True)
    judge_result_paths[label] = result_dir

    for eval_file in sorted(evaluation_dir.glob("*.json")):
        rows = read_eval_data(eval_file)
        out_path = result_dir / f"{eval_file.stem}.jsonl"
        with out_path.open("w", encoding="utf-8") as f:
            for row in tqdm(rows, desc=f"judge {label}: {eval_file.name}"):
                generated = judge_one_prompt(row)
                f.write(json.dumps({**row, "generated": generated}, ensure_ascii=False) + "\n")

    summary = aggregate_scores(result_dir)
    summary_path = Path(cfg.output_root) / f"characterbench_summary_{label}.json"
    write_json(summary_path, summary)
    summary_paths[label] = summary_path
    print(label, json.dumps(summary, indent=2, ensure_ascii=False))

unload_model(judge_model)
del judge_tokenizer


## Compare Base vs LoRA


In [ ]:
base_summary = read_json(summary_paths["base"])
lora_summary = read_json(summary_paths["lora"])
comparison = {}
for key in sorted(set(base_summary.keys()) | set(lora_summary.keys())):
    if key not in base_summary or key not in lora_summary:
        continue
    comparison[key] = {
        "base": base_summary[key],
        "lora": lora_summary[key],
        "delta": lora_summary[key] - base_summary[key],
    }

comparison_path = Path(cfg.output_root) / "characterbench_base_vs_lora_comparison.json"
write_json(comparison_path, comparison)
print(json.dumps(comparison, indent=2, ensure_ascii=False))
print("Saved comparison to:", comparison_path)
